# Agentic Design Patterns: The Refraction Pattern (Reflexion)

Welcome to the **Refraction Pattern** (more broadly known in literature as *Reflexion*).

If **Reflection** (Pattern 01) is about an agent internally critiquing its own thoughts before acting, **Refraction** is about learning from the *external world* after acting.

When an agent interacts with an external tool (like a web search, an API, or a code execution sandbox) and fails, we don't just want it to blindly try again. We want it to "refract" its trajectory based on the failure. 

In this pattern, when a tool call fails, a separate "Refraction Node" strictly analyzes *why* the tool failed, writes a discrete "Lesson Learned" summary, and injects that lesson into the agent's working memory so it does not repeat the exact same mistake on the next pass.

---

## 1. Environment Setup

In [ ]:
import os
from typing import Annotated, TypedDict, List
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

print("Refraction Environment Ready!")

## 2. Defining a Simulated "External Tool"
Imagine an API that is very strict about its input format.

In [ ]:
def strict_database_query(query_string: str) -> str:
    """Simulates a database that ONLY accepts UPPERCASE queries ending with a semicolon."""
    if not query_string.isupper():
        return "ERROR: SyntaxError. Query must be completely uppercase."
    if not query_string.endswith(";"):
        return "ERROR: SyntaxError. Query must end with a semicolon."
        
    return "SUCCESS: 42 (Database rows returned)"

## 3. Defining State & Pydantic Definitions
The agent will store its `lessons_learned` explicitly in state.

In [ ]:
class AgentAction(BaseModel):
    """The exact string to pass to the database tool."""
    query_string: str = Field(description="The query to execute.")

class LessonLearned(BaseModel):
    """An explicit note to self about what went wrong."""
    lesson: str = Field(description="A directive on how to change behavior to avoid the previous error.")

class RefractionState(TypedDict):
    messages: Annotated[List[BaseMessage], lambda a, b: a + b]
    lessons_learned: List[str]
    latest_query: str
    tool_output: str
    is_success: bool
    attempts: int

actor_llm = llm.with_structured_output(AgentAction)
refractor_llm = llm.with_structured_output(LessonLearned)

## 4. Building the Nodes
Actor -> Tool -> (If Error) -> Refractor -> Actor

In [ ]:
def actor_node(state: RefractionState):
    print(f"\n--- ACTOR (Attempt {state.get('attempts', 0) + 1}) ---")
    
    # Inject lessons learned into the prompt
    system_prompt = "You are a database querying agent.\n"
    if state.get("lessons_learned"):
        system_prompt += "\nCRITICAL PREVIOUS LESSONS:\n"
        for lesson in state["lessons_learned"]:
            system_prompt += f"- {lesson}\n"
            
    system_msg = SystemMessage(content=system_prompt)
    
    response = actor_llm.invoke([system_msg] + state["messages"])
    print(f"Actor drafts query: '{response.query_string}'")
    
    return {
        "latest_query": response.query_string,
        "attempts": state.get("attempts", 0) + 1
    }

def tool_node(state: RefractionState):
    print("--- EXECUTING TOOL ---")
    output = strict_database_query(state["latest_query"])
    print(f"Tool Output: {output}")
    
    is_success = output.startswith("SUCCESS")
    
    if is_success:
        return {"tool_output": output, "is_success": True, "messages": [AIMessage(content=f"Final output: {output}")]}
    else:
        return {"tool_output": output, "is_success": False}

def refraction_node(state: RefractionState):
    print("--- REFRACTOR ANALYZING ERROR ---")
    diagnostic_prompt = f"""
    The actor attempted to use the database tool but failed.
    Query sent: {state['latest_query']}
    Error received: {state['tool_output']}
    
    Write a highly specific 'lesson learned' instructing the actor on what rules it MUST follow next time to bypass this error.
    """
    
    response = refractor_llm.invoke([HumanMessage(content=diagnostic_prompt)])
    print(f"Lesson Derived: {response.lesson}")
    
    # Append the lesson to our memory state and inform the messages log
    return {
        "lessons_learned": state.get("lessons_learned", []) + [response.lesson],
        "messages": [HumanMessage(content=f"Tool Failed. Refractor Lesson: {response.lesson}")]
    }

def route_after_tool(state: RefractionState):
    if state["is_success"]:
        return END
    if state.get("attempts", 0) >= 4:
        print("Max attempts reached.")
        return END
    return "refractor"

## 5. Compile and Visualize

In [ ]:
builder = StateGraph(RefractionState)

builder.add_node("actor", actor_node)
builder.add_node("tool", tool_node)
builder.add_node("refractor", refraction_node)

builder.add_edge(START, "actor")
builder.add_edge("actor", "tool")
builder.add_conditional_edges("tool", route_after_tool)
builder.add_edge("refractor", "actor")

app = builder.compile()

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Graph visualization failed.")

## 6. Testing the Refraction Loop
We will ask the agent to query the user table, but we won't tell it about the strict syntax rules. Watch as it attempts, fails, *studies the error*, stores the syntax requirement in its `lessons_learned`, and successfully retrieves the data.

In [ ]:
query = "Query the user table for the count of rows."

initial_state = {
    "messages": [HumanMessage(content=query)],
    "lessons_learned": []
}

# We will stream the nodes
for chunk in app.stream(initial_state, stream_mode="updates"):
    pass # Print statements inside the nodes handle the logging

## Summary

The **Refraction Pattern** is powerful because it uses *episodic memory*.

Instead of trying the same prompt and hoping the LLM randomly generates a different string on attempt #2 (which is what standard ReAct often does when stuck), the Refraction agent halts, analyzes the physical traceback, writes down a rule, and forces the actor to read that rule on the next cycle.